# WAC Hemisphere Polar-Distortion Tiling Example

This focused copy of the polar tiling example uses the raw north- and south-hemisphere GeoTIFFs from one WAC global product family. Its primary comparison sends adjacent north-hemisphere AOIs through numbered LTM at 80–82°N and `LPS_N` at 82–84°N. A separate south-polar query remains available. The notebook first plots decimated overviews in each source raster's native equirectangular geometry, then creates and plots the routed lunar-grid cubes.

The generic `WAC_GLOBAL.iau.tif` and empirical WAC files do not index over these polar AOIs. The defaults therefore use `WAC_GLOBAL_P900N0000_100M.eqc.iau2.tif` and `WAC_GLOBAL_P900S0000_100M.eqc.iau2.tif`, the explicit polar-hemisphere products already linked into `staticLinks`. All cube warps use bilinear resampling.

### Setup

Run this notebook from the repository's top-level `notebooks/` directory with the `lfm_kernel` kernel.

## Imports and repository discovery

Repository discovery matches the supported tiling notebooks and normalizes the JupyterHub `/panfs` path to the equivalent `/explore` path before importing LFM.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import rasterio
from rasterio.enums import Resampling

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [ ]:
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists() or not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find the lfm/ and model/ directories. Run this notebook "
        "from the repository's top-level notebooks/ directory."
    )

sys.path.insert(0, str(repo_root))

from model import (
    AutomaticTilingError,
    TileAOIQuery,
    TileSourceConfig,
    TileSourceDefinition,
    create_tiles_for_query,
    default_zoom_for_modality,
)
from lfm.all_models.all_tasks.tiling_utils import RUN_ID
from lfm.all_models.all_tasks.viz import (
    display_array,
    plot_cube_records,
    print_record_summary,
    read_record_band,
    robust_limits,
)

print(f"Repository root: {repo_root}")
print("Successfully imported the automatic LFM tiling workflow")

## User configuration

`NORTH_SOURCE_TIF_PATH` and `SOUTH_SOURCE_TIF_PATH` identify the equirectangular hemisphere rasters. `SOURCE_BAND_NUMBER` is the 1-based band displayed from both raw sources and tiled outputs.

AOIs use IAU:30100 lunar latitude and longitude in `ul_lat`, `ul_lon`, `lr_lat`, `lr_lon` order. The two comparison AOIs use the same 23–25°E longitude interval, which stays inside one numbered LTM zone. The first spans 80–82°N and routes only to LTM; the second spans 82–84°N and routes only to `LPS_N` because the polar threshold is inclusive at 82°. Here “Mercator” means Armstrong's local transverse Mercator (LTM), not Web Mercator.

In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
NORTH_SOURCE_TIF_PATH = PROJECT_DATA_DIR / "staticLinks/WAC_GLOBAL_P900N0000_100M.eqc.iau2.tif"
SOUTH_SOURCE_TIF_PATH = PROJECT_DATA_DIR / "staticLinks/WAC_GLOBAL_P900S0000_100M.eqc.iau2.tif"
SOURCE_LABEL = "WAC global 100 m hemisphere mosaic"
SOURCE_BAND_NUMBER = 1  # 1-based source/output band to visualize.

NEAR_POLAR_LTM_AOI_BOUNDS = {
    "ul_lat": 82.0,
    "ul_lon": 23.0,
    "lr_lat": 80.0,
    "lr_lon": 25.0,
}

NORTH_POLAR_AOI_BOUNDS = {
    "ul_lat": 84.0,
    "ul_lon": 23.0,
    "lr_lat": 82.0,
    "lr_lon": 25.0,
}

SOUTH_POLAR_AOI_BOUNDS = {
    "ul_lat": -85.5,
    "ul_lon": 27.5,
    "lr_lat": -86.5,
    "lr_lon": 32.5,
}

LTM_ZOOM = default_zoom_for_modality("wac", "ltm")  # Approximately 75.8 m/pixel.
POLAR_ZOOM = 4  # Approximately 73.8 m/pixel in both polar grids.
MAX_PLOT_TILES = 4  # Display-only limit; all selected cubes are written.
INDEX_WORKER_COUNT = None  # None uses SLURM_CPUS_PER_TASK; 1 forces serial indexing.

## Derived paths and source validation

Each exact TIFF filename is used as its own index inventory, so other files in `staticLinks` are not scanned or stacked. The two GeoPackages are persistent per-clone caches and may be rebuilt automatically if either source path changes. Timestamped cubes and plots are written beneath `notebooks/outputs/tiling_global_polar/`.

In [ ]:
missing_source_paths = tuple(
    path
    for path in (NORTH_SOURCE_TIF_PATH, SOUTH_SOURCE_TIF_PATH)
    if not path.is_file()
)
if missing_source_paths:
    raise FileNotFoundError(f"Polar WAC source TIFFs are missing: {missing_source_paths}")

BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "tiling_global_polar"
INDEX_CACHE_DIR = BASE_OUTPUT_DIR / "indexes"
OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
NORTH_SOURCE_INDEX = INDEX_CACHE_DIR / "wac_p900n_polar_distortion_source.gpkg"
SOUTH_SOURCE_INDEX = INDEX_CACHE_DIR / "wac_p900s_polar_distortion_source.gpkg"

print(f"North source TIFF: {NORTH_SOURCE_TIF_PATH}")
print(f"South source TIFF: {SOUTH_SOURCE_TIF_PATH}")
print(f"Notebook outputs will default here: {OUTPUT_DIR}")
print(f"North source index: {NORTH_SOURCE_INDEX}")
print(f"South source index: {SOUTH_SOURCE_INDEX}")

## Preview the source geometry

This reads small, bilinearly downsampled overviews rather than loading the full hemisphere rasters. Each query AOI is outlined on its native equirectangular source image. The high-latitude source geometry is visibly stretched horizontally; the later plots show the same imagery after reprojection onto its stereographic polar tile grid.

In [ ]:
def read_source_overview(path):
    with rasterio.open(path) as source:
        scale = min(1.0, 800 / source.height, 2400 / source.width)
        overview_height = max(1, round(source.height * scale))
        overview_width = max(1, round(source.width * scale))
        overview = source.read(
            SOURCE_BAND_NUMBER,
            out_shape=(overview_height, overview_width),
            masked=True,
            resampling=Resampling.bilinear,
        )
        return {
            "array": overview,
            "bounds": source.bounds,
            "crs": source.crs,
            "shape": (source.height, source.width),
            "nodata": source.nodata,
        }

preview_specs = (
    (
        "North hemisphere",
        NORTH_SOURCE_TIF_PATH,
        (
            ("LTM query: 80–82°N", NEAR_POLAR_LTM_AOI_BOUNDS, "lime"),
            ("LPS_N query: 82–84°N", NORTH_POLAR_AOI_BOUNDS, "cyan"),
        ),
    ),
    (
        "South hemisphere",
        SOUTH_SOURCE_TIF_PATH,
        (("LPS_S query", SOUTH_POLAR_AOI_BOUNDS, "magenta"),),
    ),
)
figure, axes = plt.subplots(2, 1, figsize=(14, 11), constrained_layout=True)
for axis, (hemisphere, path, aoi_specs) in zip(axes, preview_specs):
    preview = read_source_overview(path)
    bounds = preview["bounds"]
    print(
        f"{hemisphere}: CRS={preview['crs']}; shape={preview['shape']}; "
        f"bounds={bounds}; NoData={preview['nodata']}"
    )
    vmin, vmax = robust_limits(preview["array"])
    rendered = axis.imshow(
        display_array(preview["array"]),
        cmap="gray",
        vmin=vmin,
        vmax=vmax,
        extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
    )
    axis.set_title(f"Native equirectangular source: {hemisphere}")
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    figure.colorbar(rendered, ax=axis, shrink=0.75)
    geographic_like_bounds = (
        -95 <= bounds.bottom <= 95
        and -95 <= bounds.top <= 95
        and bounds.right - bounds.left <= 370
    )
    if geographic_like_bounds:
        for label, aoi_bounds, color in aoi_specs:
            axis.add_patch(
                Rectangle(
                    (aoi_bounds["ul_lon"], aoi_bounds["lr_lat"]),
                    aoi_bounds["lr_lon"] - aoi_bounds["ul_lon"],
                    aoi_bounds["ul_lat"] - aoi_bounds["lr_lat"],
                    fill=False,
                    edgecolor=color,
                    linewidth=2,
                    label=label,
                )
            )
        axis.legend(loc="lower right")
    else:
        warnings.warn(
            f"{hemisphere} source bounds do not look geographic; "
            "the AOI rectangle was not drawn.",
            stacklevel=2,
        )

SOURCE_OVERVIEW_PATH = OUTPUT_DIR / "plots" / "hemisphere_source_overviews.png"
SOURCE_OVERVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(SOURCE_OVERVIEW_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved source overviews: {SOURCE_OVERVIEW_PATH}")

## Configure the hemisphere sources

This is a static-only diagnostic. Each raw hemisphere raster receives its own exact-match index and is enabled only for its corresponding polar query. Both use bilinear tiling and write `-32768` as output NoData.

In [ ]:
def make_hemisphere_source(source_path, index_path, zoom_overrides):
    return TileSourceDefinition(
        source=TileSourceConfig(
            name="static",
            data_dir=source_path.parent,
            index_path=index_path,
            location_field="location",
            selection_mode="all_intersecting",
            band_indices=(SOURCE_BAND_NUMBER,),
            resampling="bilinear",
            output_nodata=-32768,
            required=True,
        ),
        role="static",
        modality="custom",
        image_globs=(source_path.name,),
        zoom_overrides=zoom_overrides,
        polar_supported=True,
        rebuild_invalid_index=True,
        index_worker_count=INDEX_WORKER_COUNT,
    )

north_hemisphere_source = make_hemisphere_source(
    NORTH_SOURCE_TIF_PATH,
    NORTH_SOURCE_INDEX,
    {"ltm": LTM_ZOOM, "lps_n": POLAR_ZOOM},
)
south_polar_source = make_hemisphere_source(
    SOUTH_SOURCE_TIF_PATH,
    SOUTH_SOURCE_INDEX,
    {"lps_s": POLAR_ZOOM},
)
print(north_hemisphere_source)
print(south_polar_source)

## Near-polar LTM query: 80–82°N

This AOI ends exactly at the inclusive polar boundary and therefore remains in one numbered north LTM zone. WAC's standard LTM zoom 5 is approximately 75.8 m/pixel, close to the 73.8 m/pixel polar zoom used next.

In [ ]:
try:
    ltm_records = create_tiles_for_query(
        query=TileAOIQuery(**NEAR_POLAR_LTM_AOI_BOUNDS),
        output_dir=OUTPUT_DIR / "near_polar_ltm",
        static_sources=(north_hemisphere_source,),
        include_dynamic=False,
        include_static=True,
        stdout=sys.stdout,
    )
except AutomaticTilingError as exc:
    ltm_records = list(exc.completed_records)
    warnings.warn(
        f"Near-polar LTM query was incomplete and will not stop the polar query: {exc}",
        stacklevel=2,
    )

print_record_summary(ltm_records)
if ltm_records:
    if any(record.grid_id.startswith("LPS_") for record in ltm_records):
        raise AssertionError("The 80–82° query unexpectedly produced a polar record.")
    ltm_figure = plot_cube_records(
        ltm_records,
        source_name="static",
        source_label=f"{SOURCE_LABEL} — LTM 80–82°N",
        band_number=1,
        output_path=OUTPUT_DIR / "plots" / "near_polar_ltm_cubes.png",
        max_tiles=MAX_PLOT_TILES,
    )
else:
    ltm_figure = None
    print("No near-polar LTM cubes were available to plot.")

## Polar query: 82–84°N

No grid identifier is supplied. Because the AOI begins at the inclusive +82° threshold, automatic routing selects `LPS_N` and polar zoom 4. The same raw north-hemisphere TIFF and cached index are reused.

In [ ]:
try:
    north_records = create_tiles_for_query(
        query=TileAOIQuery(**NORTH_POLAR_AOI_BOUNDS),
        output_dir=OUTPUT_DIR / "north_polar",
        static_sources=(north_hemisphere_source,),
        include_dynamic=False,
        include_static=True,
        stdout=sys.stdout,
    )
except AutomaticTilingError as exc:
    north_records = list(exc.completed_records)
    warnings.warn(
        f"North-polar query was incomplete and will not stop the south query: {exc}",
        stacklevel=2,
    )

print_record_summary(north_records)
if north_records:
    if any(record.grid_id != "LPS_N" for record in north_records):
        raise AssertionError("North query produced a non-LPS_N record.")
    north_figure = plot_cube_records(
        north_records,
        source_name="static",
        source_label=f"{SOURCE_LABEL} — north polar",
        band_number=1,
        output_path=OUTPUT_DIR / "plots" / "north_polar_cubes.png",
        max_tiles=MAX_PLOT_TILES,
    )
else:
    north_figure = None
    print("No north-polar cubes were available to plot.")

## LTM versus polar comparison

This figure places up to four LTM tiles in the first row and four `LPS_N` tiles in the second. Both rows use one robust color scale computed from all displayed tiles, so contrast changes do not masquerade as projection changes. Tile coordinates and grid IDs remain in each title. The two AOIs are adjacent rather than identical because the routing contract assigns each latitude to exactly one grid family.

In [ ]:
comparison_groups = (
    ("LTM 80–82°N", ltm_records[:MAX_PLOT_TILES]),
    ("LPS_N 82–84°N", north_records[:MAX_PLOT_TILES]),
)
if all(records for _, records in comparison_groups):
    comparison_images = []
    finite_values = []
    for row_label, records in comparison_groups:
        row_images = []
        for record in records:
            image, band_name, band_number = read_record_band(record, band_number=1)
            row_images.append((record, image, band_name, band_number))
            values = np.asarray(image.compressed(), dtype=np.float64)
            finite_values.append(values[np.isfinite(values)])
        comparison_images.append((row_label, row_images))

    finite_chunks = [values for values in finite_values if values.size]
    if finite_chunks:
        combined_values = np.concatenate(finite_chunks)
        comparison_vmin, comparison_vmax = np.percentile(combined_values, (2.0, 98.0))
        if np.isclose(comparison_vmin, comparison_vmax):
            comparison_vmax = comparison_vmin + 1.0
    else:
        comparison_vmin, comparison_vmax = 0.0, 1.0
    comparison_columns = max(len(images) for _, images in comparison_images)
    comparison_figure, comparison_axes = plt.subplots(
        2,
        comparison_columns,
        figsize=(5 * comparison_columns, 10),
        squeeze=False,
        constrained_layout=True,
    )
    rendered = None
    for row, (row_label, images) in enumerate(comparison_images):
        for column in range(comparison_columns):
            axis = comparison_axes[row, column]
            if column >= len(images):
                axis.axis("off")
                continue
            record, image, band_name, band_number = images[column]
            rendered = axis.imshow(
                display_array(image),
                cmap="gray",
                vmin=comparison_vmin,
                vmax=comparison_vmax,
            )
            axis.set_title(
                f"{row_label} | {record.grid_id} z{record.zoom_level}\n"
                f"tile ({record.tile_x}, {record.tile_y}) | band {band_number}: {band_name}"
            )
            axis.axis("off")
    if rendered is not None:
        comparison_figure.colorbar(
            rendered, ax=comparison_axes.ravel().tolist(), shrink=0.7
        )
    comparison_figure.suptitle(
        "Adjacent WAC source regions: LTM versus north-polar tiling"
    )
    COMPARISON_PATH = OUTPUT_DIR / "plots" / "ltm_vs_lps_n_comparison.png"
    comparison_figure.savefig(COMPARISON_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved LTM/polar comparison: {COMPARISON_PATH}")
else:
    comparison_figure = None
    print("Both LTM and LPS_N records are required for the comparison plot.")

## South-polar AOI query

The second call prepares or reuses the independent south-hemisphere index. Because this AOI is entirely south of -82°, automatic routing selects `LPS_S`.

In [ ]:
try:
    south_records = create_tiles_for_query(
        query=TileAOIQuery(**SOUTH_POLAR_AOI_BOUNDS),
        output_dir=OUTPUT_DIR / "south_polar",
        static_sources=(south_polar_source,),
        include_dynamic=False,
        include_static=True,
        stdout=sys.stdout,
    )
except AutomaticTilingError as exc:
    south_records = list(exc.completed_records)
    warnings.warn(
        f"South-polar query was incomplete: {exc}",
        stacklevel=2,
    )

print_record_summary(south_records)
if south_records:
    if any(record.grid_id != "LPS_S" for record in south_records):
        raise AssertionError("South query produced a non-LPS_S record.")
    south_figure = plot_cube_records(
        south_records,
        source_name="static",
        source_label=f"{SOURCE_LABEL} — south polar",
        band_number=1,
        output_path=OUTPUT_DIR / "plots" / "south_polar_cubes.png",
        max_tiles=MAX_PLOT_TILES,
    )
else:
    south_figure = None
    print("No south-polar cubes were available to plot.")

## Outputs

The output directory contains paired native-source overviews, individual LTM and polar cube plots, a shared-scale `ltm_vs_lps_n_comparison.png`, the optional south-polar plot, and all routed GeoTIFF cubes. Two reusable single-source GeoPackage indexes live in the persistent index cache. Use the shared-scale comparison as the primary view of how adjacent high-latitude WAC imagery changes between Armstrong LTM and polar stereographic tiling.

In [ ]:
print(f"Completed global polar comparison: {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}")